## Import packages

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
from src import create_data_generators, get_class_weights
from src import build_transfer_model
from src import train_model
from src import evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_training_history

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Load data with augmentation

In [ ]:
data_dir = '../data/chest_xray'
train_gen, val_gen, test_gen = create_data_generators(
    data_dir, target_size=(224, 224), batch_size=32, augment=True
)

## Get class weights

In [ ]:
class_weights = get_class_weights(train_gen)

## Build transfer learning model (frozen base)

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model, base_model = build_transfer_model(
    input_shape=(224, 224, 3),
    base_model_name="resnet50",
    learning_rate=0.0001,
    fine_tune_layers=0  # All base layers frozen
)

model.summary()

## Phase 1 — Train with frozen base

In [ ]:
# Only the custom classification head is trained
# The pre-trained ResNet50 layers stay frozen
print("PHASE 1: Training classification head only")
print("=" * 40)

history_phase1 = train_model(
    model,
    train_gen, val_gen,
    epochs=15,
    class_weights=class_weights,
    model_save_path='../models/transfer_phase1.keras'
)

## Plot Phase 1 training history

In [ ]:
plot_training_history(history_phase1, title="Phase 1 - Frozen Base")

## Evaluate Phase 1

In [ ]:
results_phase1 = evaluate_model(model, test_gen)
print("\nPhase 1 complete — classification head trained")

## Phase 2 — Fine-tune top layers

In [ ]:
# Unfreeze the last 20 layers of ResNet50 for fine-tuning
# This lets the model slightly adjust pre-trained features
# to better match chest X-ray patterns
print("\nPHASE 2: Fine-tuning top layers")
print("=" * 40)

base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable = sum(1 for layer in model.layers if layer.trainable)
frozen = sum(1 for layer in model.layers if not layer.trainable)
print(f"Trainable layers: {trainable}, Frozen layers: {frozen}")

# Recompile with a lower learning rate for fine-tuning
# Lower LR prevents destroying pre-trained weights
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Train Phase 2

In [ ]:
history_phase2 = train_model(
    model,
    train_gen, val_gen,
    epochs=15,
    class_weights=class_weights,
    model_save_path='../models/transfer_model.keras'
)

## Plot Phase 2 training history

In [ ]:
plot_training_history(history_phase2, title="Phase 2 - Fine-Tuned")

## Evaluate fine-tuned model on test set

In [ ]:
results = evaluate_model(model, test_gen)

## Confusion matrix

In [ ]:
plot_confusion_matrix(
    results['y_true'], results['y_pred'],
    results['class_names'],
    title="Transfer Learning (ResNet50) - Confusion Matrix"
)

## ROC curve

In [ ]:
auc_score = plot_roc_curve(
    results['y_true'], results['y_prob'],
    title="Transfer Learning (ResNet50) - ROC Curve"
)

## Compare with baseline

In [ ]:
with open('../models/baseline_results.json', 'r') as f:
    baseline = json.load(f)

print("\nBASELINE vs TRANSFER LEARNING:")
print(f"{'Metric':<12} {'Baseline':<12} {'Transfer':<12} {'Improvement':<12}")
print("-" * 48)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    base_val = baseline[metric]
    trans_val = results[metric] if metric != 'auc' else auc_score
    diff = trans_val - base_val
    print(f"{metric:<12} {base_val:<12.4f} {trans_val:<12.4f} {diff:+.4f}")


## Save best model

In [ ]:
transfer_results = {
    "model": "transfer_resnet50",
    "accuracy": results['accuracy'],
    "precision": results['precision'],
    "recall": results['recall'],
    "f1": results['f1'],
    "auc": auc_score
}

with open('../models/transfer_results.json', 'w') as f:
    json.dump(transfer_results, f, indent=2)

# Auto-select best model
import shutil

if results['accuracy'] > baseline['accuracy']:
    print("\nTransfer model is better — saving as best model")
    shutil.copy('../models/transfer_model.keras', '../models/best_model.keras')
    best_source = "transfer_resnet50"
else:
    print("\nBaseline model is better — saving as best model")
    shutil.copy('../models/baseline_model.keras', '../models/best_model.keras')
    best_source = "baseline_cnn"

best_config = {
    "source": best_source,
    "accuracy": max(results['accuracy'], baseline['accuracy'])
}

with open('../models/best_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)

print(f"Best model: {best_source}")
print("Saved to: ../models/best_model.keras")

## Summary

In [ ]:
print("=" * 50)
print("TRANSFER LEARNING SUMMARY")
print("=" * 50)
print(f"Base model: ResNet50")
print(f"Phase 1 (frozen): {results_phase1['accuracy'] * 100:.2f}% accuracy")
print(f"Phase 2 (fine-tuned): {results['accuracy'] * 100:.2f}% accuracy")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")
print(f"AUC:       {auc_score:.4f}")
print("=" * 50)
print("\nNext: 05_evaluation.ipynb")